# Module 2.2: Multi-Tool Agents and Tool Chaining

## Learning Objectives

By completing this notebook, you will:
1. Build agents that coordinate multiple tools to solve complex tasks
2. Implement tool chaining where outputs feed into subsequent tools
3. Handle tool dependencies and execution ordering
4. Create specialized tool libraries for different domains
5. Debug and trace multi-step tool execution

## Prerequisites

- Module 2.1 completed (basic tool use)
- Understanding of agent loops
- Familiarity with async programming (helpful but not required)

## Estimated Time: 70 minutes

---

## 1. Setup & Imports

We'll extend our previous agent implementation to handle more complex multi-tool scenarios.

In [ ]:
# Install required packages
# !pip install anthropic requests

import json
import os
from typing import Dict, List, Any, Callable, Optional, Tuple
from datetime import datetime
import anthropic
from dataclasses import dataclass, field
import requests

print("✓ Imports successful")

## 2. Tool Execution Tracking

For complex multi-tool workflows, we need to track:
- Which tools were called
- In what order
- What parameters were used
- What results were returned

This creates an **execution trace** for debugging and analysis.

In [ ]:
# Purpose: Track tool execution for debugging and analysis
# Key Concept: Execution traces help understand agent decision-making

@dataclass
class ToolCall:
    """Record of a single tool execution."""
    tool_name: str
    inputs: Dict[str, Any]
    output: Dict[str, Any]
    timestamp: str = field(default_factory=lambda: datetime.now().isoformat())
    execution_time_ms: Optional[float] = None
    
    def to_dict(self) -> Dict:
        return {
            "tool": self.tool_name,
            "inputs": self.inputs,
            "output": self.output,
            "timestamp": self.timestamp,
            "execution_time_ms": self.execution_time_ms
        }

@dataclass
class ExecutionTrace:
    """Complete trace of agent execution."""
    user_query: str
    tool_calls: List[ToolCall] = field(default_factory=list)
    final_response: Optional[str] = None
    total_iterations: int = 0
    
    def add_tool_call(self, call: ToolCall) -> None:
        """Add a tool call to the trace."""
        self.tool_calls.append(call)
    
    def summary(self) -> str:
        """Generate human-readable summary."""
        summary = f"Query: {self.user_query}\n"
        summary += f"Iterations: {self.total_iterations}\n"
        summary += f"Tools Used: {len(self.tool_calls)}\n\n"
        
        for i, call in enumerate(self.tool_calls, 1):
            summary += f"{i}. {call.tool_name}({json.dumps(call.inputs)})\n"
            summary += f"   → {call.output}\n"
        
        summary += f"\nFinal Response: {self.final_response}"
        return summary

# Example trace
trace = ExecutionTrace(user_query="What's 10 + 5?")
trace.add_tool_call(ToolCall(
    tool_name="calculator",
    inputs={"operation": "add", "a": 10, "b": 5},
    output={"success": True, "result": 15}
))
trace.final_response = "The sum of 10 and 5 is 15."
trace.total_iterations = 1

print(trace.summary())

## 3. Enhanced Multi-Tool Agent

We'll upgrade our agent to:
- Support many tools
- Track execution traces
- Handle tool chaining automatically
- Provide better error messages

In [ ]:
# Purpose: Build production-ready multi-tool agent
# Key Concept: Agent manages tool orchestration, execution, and result synthesis

class MultiToolAgent:
    """Agent that can use multiple tools and chain them together."""
    
    def __init__(self, api_key: Optional[str] = None, verbose: bool = True):
        """
        Initialize multi-tool agent.
        
        Parameters
        ----------
        api_key : str, optional
            Anthropic API key
        verbose : bool
            Whether to print execution details
        """
        self.client = anthropic.Anthropic(
            api_key=api_key or os.environ.get("ANTHROPIC_API_KEY")
        )
        self.tools = {}
        self.tool_schemas = []
        self.verbose = verbose
        self.current_trace: Optional[ExecutionTrace] = None
    
    def register_tool(self, schema: Dict, implementation: Callable) -> None:
        """Register a tool with schema and implementation."""
        tool_name = schema["name"]
        self.tools[tool_name] = implementation
        self.tool_schemas.append(schema)
        if self.verbose:
            print(f"✓ Registered: {tool_name}")
    
    def execute_tool(self, tool_name: str, tool_input: Dict) -> Dict:
        """Execute tool and record in trace."""
        import time
        
        if tool_name not in self.tools:
            return {"success": False, "error": f"Unknown tool: {tool_name}"}
        
        start_time = time.time()
        
        try:
            result = self.tools[tool_name](**tool_input)
            execution_time = (time.time() - start_time) * 1000
            
            # Record in trace
            if self.current_trace:
                self.current_trace.add_tool_call(ToolCall(
                    tool_name=tool_name,
                    inputs=tool_input,
                    output=result,
                    execution_time_ms=execution_time
                ))
            
            return result
        
        except Exception as e:
            error_result = {"success": False, "error": f"Execution failed: {str(e)}"}
            
            if self.current_trace:
                self.current_trace.add_tool_call(ToolCall(
                    tool_name=tool_name,
                    inputs=tool_input,
                    output=error_result
                ))
            
            return error_result
    
    def run(
        self,
        user_message: str,
        max_iterations: int = 10,
        return_trace: bool = False
    ) -> str | Tuple[str, ExecutionTrace]:
        """
        Run agent with user message.
        
        Parameters
        ----------
        user_message : str
            User's query
        max_iterations : int
            Maximum tool use iterations
        return_trace : bool
            Whether to return execution trace
        
        Returns
        -------
        str or tuple
            Final response, optionally with trace
        """
        # Initialize trace
        self.current_trace = ExecutionTrace(user_query=user_message)
        
        messages = [{"role": "user", "content": user_message}]
        
        for iteration in range(max_iterations):
            if self.verbose:
                print(f"\n{'='*60}")
                print(f"Iteration {iteration + 1}")
                print("="*60)
            
            # Call Claude
            response = self.client.messages.create(
                model="claude-3-5-sonnet-20241022",
                max_tokens=2048,
                tools=self.tool_schemas,
                messages=messages
            )
            
            if self.verbose:
                print(f"Stop reason: {response.stop_reason}")
            
            # Handle tool use
            if response.stop_reason == "tool_use":
                # Process all tool use requests in this response
                tool_results = []
                
                for block in response.content:
                    if block.type == "tool_use":
                        if self.verbose:
                            print(f"\nTool: {block.name}")
                            print(f"Input: {json.dumps(block.input, indent=2)}")
                        
                        # Execute tool
                        result = self.execute_tool(block.name, block.input)
                        
                        if self.verbose:
                            print(f"Output: {json.dumps(result, indent=2)}")
                        
                        # Store result
                        tool_results.append({
                            "type": "tool_result",
                            "tool_use_id": block.id,
                            "content": json.dumps(result)
                        })
                
                # Add assistant response and tool results to conversation
                messages.append({"role": "assistant", "content": response.content})
                messages.append({"role": "user", "content": tool_results})
                
                continue
            
            # Extract final response
            final_text = ""
            for block in response.content:
                if hasattr(block, "text"):
                    final_text += block.text
            
            # Update trace
            self.current_trace.final_response = final_text
            self.current_trace.total_iterations = iteration + 1
            
            if return_trace:
                return final_text, self.current_trace
            return final_text
        
        # Max iterations reached
        error_msg = f"Max iterations ({max_iterations}) reached"
        self.current_trace.final_response = error_msg
        self.current_trace.total_iterations = max_iterations
        
        if return_trace:
            return error_msg, self.current_trace
        return error_msg

print("MultiToolAgent class defined")

## 4. Building a Tool Library

Let's create a rich library of tools that can work together. We'll implement:
- **Data tools**: Store and retrieve information
- **Math tools**: Calculations beyond basic arithmetic
- **Web tools**: Fetch data from the internet (mock for this example)
- **Text tools**: Process and analyze text

In [ ]:
# Purpose: Create comprehensive tool library for agents
# Key Concept: Rich tool sets enable complex task solving

# DATA STORAGE TOOLS
# Simple in-memory key-value store
_memory_store = {}

def create_memory_tools() -> List[Tuple[Dict, Callable]]:
    """Create tools for storing and retrieving data."""
    
    def store_value(key: str, value: str) -> Dict:
        """Store a value in memory."""
        _memory_store[key] = value
        return {
            "success": True,
            "message": f"Stored '{value}' under key '{key}'",
            "key": key
        }
    
    def retrieve_value(key: str) -> Dict:
        """Retrieve a value from memory."""
        if key in _memory_store:
            return {
                "success": True,
                "key": key,
                "value": _memory_store[key]
            }
        return {
            "success": False,
            "error": f"No value stored for key: {key}"
        }
    
    return [
        ({
            "name": "store",
            "description": "Store a value in memory with a key. Use this to remember information for later.",
            "input_schema": {
                "type": "object",
                "properties": {
                    "key": {"type": "string", "description": "The key to store under"},
                    "value": {"type": "string", "description": "The value to store"}
                },
                "required": ["key", "value"]
            }
        }, store_value),
        ({
            "name": "retrieve",
            "description": "Retrieve a previously stored value by key.",
            "input_schema": {
                "type": "object",
                "properties": {
                    "key": {"type": "string", "description": "The key to retrieve"}
                },
                "required": ["key"]
            }
        }, retrieve_value)
    ]

# MATH TOOLS
def create_math_tools() -> List[Tuple[Dict, Callable]]:
    """Create advanced math tools."""
    
    def calculate_percentage(value: float, percentage: float) -> Dict:
        """Calculate percentage of a value."""
        result = (value * percentage) / 100
        return {
            "success": True,
            "result": result,
            "calculation": f"{percentage}% of {value} = {result}"
        }
    
    def list_sum(numbers: List[float]) -> Dict:
        """Sum a list of numbers."""
        total = sum(numbers)
        return {
            "success": True,
            "result": total,
            "count": len(numbers),
            "average": total / len(numbers) if numbers else 0
        }
    
    return [
        ({
            "name": "percentage",
            "description": "Calculate what percentage of a value is. For example, 20% of 100.",
            "input_schema": {
                "type": "object",
                "properties": {
                    "value": {"type": "number", "description": "The base value"},
                    "percentage": {"type": "number", "description": "The percentage"}
                },
                "required": ["value", "percentage"]
            }
        }, calculate_percentage),
        ({
            "name": "sum_list",
            "description": "Sum a list of numbers and get count and average.",
            "input_schema": {
                "type": "object",
                "properties": {
                    "numbers": {
                        "type": "array",
                        "items": {"type": "number"},
                        "description": "List of numbers to sum"
                    }
                },
                "required": ["numbers"]
            }
        }, list_sum)
    ]

# WEB TOOLS (mock)
def create_web_tools() -> List[Tuple[Dict, Callable]]:
    """Create mock web interaction tools."""
    
    def mock_search(query: str) -> Dict:
        """Mock search (returns simulated results)."""
        # In production, this would call a real search API
        return {
            "success": True,
            "query": query,
            "results": [
                f"Result 1 for '{query}': Simulated search result",
                f"Result 2 for '{query}': Another simulated result"
            ],
            "note": "This is a mock search for demonstration"
        }
    
    return [
        ({
            "name": "search",
            "description": "Search the web for information. Use when you need current or external information.",
            "input_schema": {
                "type": "object",
                "properties": {
                    "query": {"type": "string", "description": "Search query"}
                },
                "required": ["query"]
            }
        }, mock_search)
    ]

print("Tool library created:")
print("  - Memory tools (store, retrieve)")
print("  - Math tools (percentage, sum_list)")
print("  - Web tools (search)")

## 5. Building and Testing Multi-Tool Agent

Now let's create an agent with all these tools and test it on complex tasks that require tool chaining.

In [ ]:
# Purpose: Create fully-equipped agent and test multi-tool workflows
# Key Concept: Tool chaining emerges from LLM reasoning

# Create agent
agent = MultiToolAgent(verbose=True)

# Register all tools
print("Registering tools...\n")

for schema, impl in create_memory_tools():
    agent.register_tool(schema, impl)

for schema, impl in create_math_tools():
    agent.register_tool(schema, impl)

for schema, impl in create_web_tools():
    agent.register_tool(schema, impl)

print(f"\n✓ Agent ready with {len(agent.tools)} tools")

In [ ]:
# Test 1: Tool chaining (store, then retrieve)
print("\n" + "#"*60)
print("TEST 1: Memory + Retrieval")
print("#"*60)

response1, trace1 = agent.run(
    "Store the number 42 under the key 'answer'. Then retrieve it and tell me what it is.",
    return_trace=True
)

print("\n" + "="*60)
print("EXECUTION TRACE")
print("="*60)
print(trace1.summary())

In [ ]:
# Test 2: Multi-step calculation
print("\n" + "#"*60)
print("TEST 2: Complex Math Workflow")
print("#"*60)

response2, trace2 = agent.run(
    "Calculate 20% of 500, then add that result to the numbers [10, 20, 30] and sum them all.",
    return_trace=True
)

print("\n" + "="*60)
print("EXECUTION TRACE")
print("="*60)
print(trace2.summary())

## Hands-On Exercises

Build your own multi-tool workflows.

### Exercise 2.2.1: Create a Statistics Tool

**Task**: Implement a tool that calculates min, max, mean, and median of a list of numbers.

**Expected Output**: Tool returns dict with all four statistics.

**Hints**:
<details>
<summary>Hint 1</summary>
Use Python's built-in min(), max(), sum(), and sorted() functions.
</details>

<details>
<summary>Hint 2</summary>
For median: sort the list, take middle element (or average of two middle for even length).
</details>

In [ ]:
# YOUR CODE HERE
# ---------------

def create_stats_tool() -> Tuple[Dict, Callable]:
    """
    Create statistics tool.
    
    Returns
    -------
    tuple
        (schema, implementation)
    """
    # TODO: Define schema and implementation
    pass

exercise_2_2_1_answer = create_stats_tool  # Don't change

In [ ]:
# AUTO-GRADED TESTS - Do not modify
# ----------------------------------

def test_exercise_2_2_1():
    schema, impl = exercise_2_2_1_answer()
    
    assert isinstance(schema, dict), "Schema should be a dict"
    assert "name" in schema, "Schema needs name"
    assert "input_schema" in schema, "Schema needs input_schema"
    
    # Test implementation
    result = impl([1, 2, 3, 4, 5])
    assert isinstance(result, dict), "Should return dict"
    assert "min" in result or "minimum" in result, "Should include min"
    assert "max" in result or "maximum" in result, "Should include max"
    assert "mean" in result or "average" in result, "Should include mean"
    assert "median" in result, "Should include median"
    
    print("✓ Exercise 2.2.1 passed!")

test_exercise_2_2_1()

### Exercise 2.2.2: Build a Multi-Step Calculator

**Task**: Create a query that requires the agent to use at least 3 different tools.

**Expected Output**: Query string that will trigger multiple tool calls.

**Example**: "Calculate 15% of 200, store it as 'discount', sum the list [50, 30, 20], then retrieve the discount."

**Hints**:
<details>
<summary>Hint 1</summary>
Combine percentage, store, sum_list, and retrieve tools.
</details>

In [ ]:
# YOUR CODE HERE
# ---------------

def create_complex_query() -> str:
    """
    Create a query that requires 3+ tool calls.
    
    Returns
    -------
    str
        Complex query
    """
    # TODO: Write a query that uses at least 3 tools
    query = ""
    return query

exercise_2_2_2_answer = create_complex_query  # Don't change

In [ ]:
# AUTO-GRADED TESTS - Do not modify
# ----------------------------------

def test_exercise_2_2_2():
    query = exercise_2_2_2_answer()
    assert isinstance(query, str), "Should return a string"
    assert len(query) > 20, "Query should be substantive (>20 chars)"
    
    # Try to run it (if agent available)
    if 'agent' in globals():
        print("Running your query...")
        response, trace = agent.run(query, return_trace=True)
        print(f"\nTools used: {len(trace.tool_calls)}")
        assert len(trace.tool_calls) >= 3, f"Query should use 3+ tools, used {len(trace.tool_calls)}"
    
    print("✓ Exercise 2.2.2 passed!")

test_exercise_2_2_2()

### Exercise 2.2.3: Trace Analyzer

**Task**: Write a function that analyzes an ExecutionTrace and returns insights.

**Expected Output**: Function returns dict with: most_used_tool, total_tools_called, avg_execution_time.

**Hints**:
<details>
<summary>Hint 1</summary>
Use Counter from collections to count tool usage.
</details>

In [ ]:
# YOUR CODE HERE
# ---------------

from collections import Counter

def analyze_trace(trace: ExecutionTrace) -> Dict[str, Any]:
    """
    Analyze execution trace.
    
    Parameters
    ----------
    trace : ExecutionTrace
        The trace to analyze
    
    Returns
    -------
    dict
        Analysis with most_used_tool, total_tools_called, avg_execution_time_ms
    """
    # TODO: Analyze the trace
    pass

exercise_2_2_3_answer = analyze_trace  # Don't change

In [ ]:
# AUTO-GRADED TESTS - Do not modify
# ----------------------------------

def test_exercise_2_2_3():
    # Create test trace
    test_trace = ExecutionTrace(user_query="test")
    test_trace.add_tool_call(ToolCall("tool_a", {}, {}, execution_time_ms=10.0))
    test_trace.add_tool_call(ToolCall("tool_a", {}, {}, execution_time_ms=20.0))
    test_trace.add_tool_call(ToolCall("tool_b", {}, {}, execution_time_ms=30.0))
    
    result = exercise_2_2_3_answer(test_trace)
    
    assert isinstance(result, dict), "Should return dict"
    assert "most_used_tool" in result, "Should identify most used tool"
    assert "total_tools_called" in result, "Should count total tools"
    assert result["total_tools_called"] == 3, "Should count 3 tool calls"
    assert result["most_used_tool"] == "tool_a", "tool_a was called twice"
    
    print("✓ Exercise 2.2.3 passed!")

test_exercise_2_2_3()

## Solutions

Reference implementations.

In [ ]:
# SOLUTION 2.2.1: Statistics Tool

def create_stats_tool_solution() -> Tuple[Dict, Callable]:
    def calculate_stats(numbers: List[float]) -> Dict:
        if not numbers:
            return {"success": False, "error": "Empty list"}
        
        sorted_nums = sorted(numbers)
        n = len(sorted_nums)
        median = sorted_nums[n//2] if n % 2 == 1 else (sorted_nums[n//2-1] + sorted_nums[n//2]) / 2
        
        return {
            "success": True,
            "min": min(numbers),
            "max": max(numbers),
            "mean": sum(numbers) / len(numbers),
            "median": median,
            "count": len(numbers)
        }
    
    schema = {
        "name": "statistics",
        "description": "Calculate min, max, mean, and median of a list of numbers.",
        "input_schema": {
            "type": "object",
            "properties": {
                "numbers": {
                    "type": "array",
                    "items": {"type": "number"},
                    "description": "List of numbers"
                }
            },
            "required": ["numbers"]
        }
    }
    
    return schema, calculate_stats

# SOLUTION 2.2.2: Complex Query

def create_complex_query_solution() -> str:
    return "Calculate 25% of 800, store it under 'tax', sum the list [100, 200, 300], and then retrieve the tax value."

# SOLUTION 2.2.3: Trace Analyzer

def analyze_trace_solution(trace: ExecutionTrace) -> Dict[str, Any]:
    if not trace.tool_calls:
        return {
            "most_used_tool": None,
            "total_tools_called": 0,
            "avg_execution_time_ms": 0
        }
    
    tool_counts = Counter(call.tool_name for call in trace.tool_calls)
    most_used = tool_counts.most_common(1)[0][0]
    
    execution_times = [call.execution_time_ms for call in trace.tool_calls if call.execution_time_ms]
    avg_time = sum(execution_times) / len(execution_times) if execution_times else 0
    
    return {
        "most_used_tool": most_used,
        "total_tools_called": len(trace.tool_calls),
        "avg_execution_time_ms": avg_time,
        "tool_usage": dict(tool_counts)
    }

## Summary

**Key Takeaways**:

1. **Tool Chaining**: LLMs automatically chain tools when needed for complex tasks
2. **Execution Traces**: Tracking tool calls helps debug and understand agent behavior
3. **Rich Tool Libraries**: More tools = more capable agents, but also more complexity
4. **Emergent Behavior**: Complex workflows emerge from simple tool definitions
5. **Error Resilience**: Tools should handle errors gracefully and return useful messages

**What's Next**:
- Module 2.3: Tool security and sandboxing
- Module 4: ReAct agents with explicit reasoning
- Module 5: Memory and state management

**Additional Resources**:
- [LangChain Tools Documentation](https://python.langchain.com/docs/modules/agents/tools/)
- [Anthropic Tool Use Best Practices](https://docs.anthropic.com/claude/docs/tool-use)

---

You can now build sophisticated multi-tool agents that solve complex tasks through tool coordination!